# Ch.2 — Neural Networks
**Track:** ML from Scratch · California Housing dataset

## Core Idea

A neural network stacks **linear → activation** blocks. Each block re-represents the data until the final prediction is (approximately) linear in the last hidden layer.

```
input x → [W₁·x + b₁ → ReLU] → [W₂·h¹ + b₂ → ReLU] → [W₃·h² + b₃] → ŷ
```

Key decisions: **activation function**, **weight initialisation**, **depth**, **width**.

## Running Example

Real estate platform — predict `MedHouseVal` (median house value, $100k units) from **all 8 features**.

| Feature | Description |
|---|---|
| MedInc | Median income (×$10k) |
| HouseAge | Median house age |
| AveRooms | Average rooms/house |
| AveBedrms | Avg bedrooms/house |
| Population | Block population |
| AveOccup | Avg occupancy |
| Latitude | Block latitude |
| Longitude | Block longitude |

Output neuron: **linear** (no activation) — house value is unbounded and continuous.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `housing` using `fetch_california_housing()`
# 3. Call `train_test_split()` to produce the result
# 4. Fit the model -- call `StandardScaler()`
# 5. Process data
#
# Hint:
#    scaler = StandardScaler(???)
#    X_train_s = scaler.fit_transform(???)
#    X_test_s = scaler.transform(???)
#    scaler.transform(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Load data
housing = fetch_california_housing()
X, y = housing.data, housing.target          # (20640, 8), (20640,)
feature_names = housing.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardise — critical for neural nets
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}] ($100k units)")

## Activation Functions — Math & Shapes

| Activation | Formula | Output range | Best for |
|---|---|---|---|
| ReLU | $\max(0, z)$ | $[0, \infty)$ | hidden layers (default) |
| Sigmoid | $\frac{1}{1+e^{-z}}$ | $(0,1)$ | binary output |
| Tanh | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1,1)$ | hidden (zero-centred) |
| Softmax | $\frac{e^{z_k}}{\sum_j e^{z_j}}$ | $(0,1)$, sums to 1 | multi-class output |
| Linear | $z$ | $(-\infty, \infty)$ | regression output |

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `z` using `linspace()`
# 2. Compute `activations` using `maximum()`
# 3. Plot results -- call `subplots()`
# 4. Plot results -- call `set_ylabel()`
#
# Hint:
#    z = np.linspace(???)
#    axes = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Plot all activation functions side by side
z = np.linspace(-4, 4, 300)

activations = {
    'ReLU':    np.maximum(0, z),
    'Sigmoid': 1 / (1 + np.exp(-z)),
    'Tanh':    np.tanh(z),
    'Linear':  z,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, vals) in zip(axes, activations.items()):
    ax.plot(z, vals, linewidth=2)
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.axvline(0, color='grey', linewidth=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('z (pre-activation)')
    ax.set_xlim(-4, 4)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('g(z)')
plt.suptitle('Activation Functions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Weight Initialisation — Xavier vs He

**Xavier / Glorot** (for Sigmoid / Tanh):
$$W \sim \mathcal{U}\!\left(-\sqrt{\frac{6}{n_\text{in}+n_\text{out}}},\ \sqrt{\frac{6}{n_\text{in}+n_\text{out}}}\right)$$

**He** (for ReLU):
$$W \sim \mathcal{N}\!\left(0,\ \sqrt{\frac{2}{n_\text{in}}}\right)$$

Both are calibrated so the **variance of activations stays roughly constant** across layers — preventing signal from exploding or vanishing before any gradient update.

In [ ]:
def forward_variance(x, layer_sizes, init='he', activation=np.tanh):
    """
    TODO #3: Implement `forward_variance()`.

    Steps:
    1. Compute `rng` using `default_rng()`
    2. Define helper function `forward_variance()`
    3. Compute `layer_labels`
    4. Plot results -- call `init()`
    5. Plot results -- call `maximum()`
    6. Plot results -- call `bar()`
    7. Plot results -- call `suptitle()`

    Hint:
    rng = np.random.default_rng(???)
    x_demo = rng.normal(???)
    W = rng.normal(???)
    limit = np.sqrt(???)

    Returns: stds
    """
    raise NotImplementedError("TODO: implement forward_variance()")

## Manual Forward Pass (NumPy)

Build the full forward pass from scratch with He-initialised weights and ReLU hidden layers.
Output layer: **linear** (regression).

In [ ]:
def relu(z):
    """
    TODO #4: Implement `relu()`.

    Steps:
    1. Define helper function `relu()`
    2. Define helper function `he_init()`
    3. Compute `rng` using `default_rng()`
    4. Compute `W1` using `zeros()`
    5. Define helper function `forward()`
    6. Compute `y_hat_init` using `Predictions()`

    Hint:
    rng = np.random.default_rng(???)
    b1 = np.zeros(???)
    b2 = np.zeros(???)
    b3 = np.zeros(???)

    Returns: np.maximum(0, z)
    """
    raise NotImplementedError("TODO: implement relu()")

def he_init(n_in, n_out, rng):
    """
    TODO #4: Implement `he_init()`.

    Steps:
    1. Define helper function `relu()`
    2. Define helper function `he_init()`
    3. Compute `rng` using `default_rng()`
    4. Compute `W1` using `zeros()`
    5. Define helper function `forward()`
    6. Compute `y_hat_init` using `Predictions()`

    Hint:
    rng = np.random.default_rng(???)
    b1 = np.zeros(???)
    b2 = np.zeros(???)
    b3 = np.zeros(???)

    Returns: np.maximum(0, z)
    """
    raise NotImplementedError("TODO: implement he_init()")

def forward(X):
    """
    TODO #4: Implement `forward()`.

    Steps:
    1. Define helper function `relu()`
    2. Define helper function `he_init()`
    3. Compute `rng` using `default_rng()`
    4. Compute `W1` using `zeros()`
    5. Define helper function `forward()`
    6. Compute `y_hat_init` using `Predictions()`

    Hint:
    rng = np.random.default_rng(???)
    b1 = np.zeros(???)
    b2 = np.zeros(???)
    b3 = np.zeros(???)

    Returns: np.maximum(0, z)
    """
    raise NotImplementedError("TODO: implement forward()")

## Baseline Comparison — Linear Regression vs Neural Net

Before tuning the network, establish how much linear regression gets us.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Fit the model -- call `predict()`
# 2. Fit the model -- call `MLPRegressor()`
# 3. Call `MLP()` to produce the result
#
# Hint:
#    lr = LinearRegression(???)
#    mlp = MLPRegressor(hidden_layer_sizes=???)
#    mlp.fit(???)
#    lr.predict(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Baseline: Linear Regression (Ch.1)
lr = LinearRegression().fit(X_train_s, y_train)
r2_lr = r2_score(y_test, lr.predict(X_test_s))
rmse_lr = mean_squared_error(y_test, lr.predict(X_test_s), squared=False)

# Neural Network: sklearn MLPRegressor (2 hidden layers, 128 + 64)
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
)
mlp.fit(X_train_s, y_train)
r2_mlp = r2_score(y_test, mlp.predict(X_test_s))
rmse_mlp = mean_squared_error(y_test, mlp.predict(X_test_s), squared=False)

print(f"{'Model':<25} {'R²':>8} {'RMSE':>10}")
print("-" * 45)
print(f"{'Linear Regression':<25} {r2_lr:>8.4f} {rmse_lr:>10.4f}")
print(f"{'MLP (128→64→1)':<25} {r2_mlp:>8.4f} {rmse_mlp:>10.4f}")
print(f"\nGain: ΔR² = {r2_mlp - r2_lr:+.4f}")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Plot results -- call `curve()`
# 2. Plot results -- call `plot()`
# 3. Plot results -- call `MedHouseVal()`
# 4. Plot results -- call `tight_layout()`
#
# Hint:
#    axes = plt.subplots(???)
#    y_pred_mlp = mlp.predict(???)
#    mx = y_test.min(???)
#    mlp.predict(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Training loss curve (MLPRegressor stores loss_curve_)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
axes[0].plot(mlp.loss_curve_, label='Training loss', color='steelblue')
if hasattr(mlp, 'validation_scores_'):
    # validation_scores_ is R², negate to make it a loss proxy for comparison
    axes[0].plot(
        [-s for s in mlp.validation_scores_],
        label='Validation loss (−R²)', color='coral', linestyle='--'
    )
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Predicted vs Actual scatter
y_pred_mlp = mlp.predict(X_test_s)
axes[1].scatter(y_test, y_pred_mlp, alpha=0.3, s=10, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual MedHouseVal ($100k)')
axes[1].set_ylabel('Predicted MedHouseVal ($100k)')
axes[1].set_title(f'Predicted vs Actual (R²={r2_mlp:.3f})')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Depth vs Width Sweep

How do different architectures compare? We test width (units per layer, assuming 2 equal layers) and depth (number of layers at fixed width 64).

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Fit the model -- call `MLPRegressor()`
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `plot()`
# 4. Plot results -- call `suptitle()`
# 5. Process data
#
# Hint:
#    m = MLPRegressor(hidden_layer_sizes=???)
#    axes = plt.subplots(???)
#    m.predict(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Width sweep: 2 hidden layers, varying units
widths = [8, 16, 32, 64, 128, 256]
width_r2 = []
for w in widths:
    m = MLPRegressor(
        hidden_layer_sizes=(w, w), activation='relu',
        solver='adam', max_iter=300, random_state=42
    ).fit(X_train_s, y_train)
    width_r2.append(r2_score(y_test, m.predict(X_test_s)))

# Depth sweep: 1–5 hidden layers of 64 units each
depths = [1, 2, 3, 4, 5]
depth_r2 = []
for d in depths:
    m = MLPRegressor(
        hidden_layer_sizes=tuple([64] * d), activation='relu',
        solver='adam', max_iter=300, random_state=42
    ).fit(X_train_s, y_train)
    depth_r2.append(r2_score(y_test, m.predict(X_test_s)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(widths, width_r2, marker='o', color='steelblue')
axes[0].set_xscale('log', base=2)
axes[0].set_xticks(widths)
axes[0].set_xticklabels(widths)
axes[0].set_xlabel('Units per layer (2 hidden layers)')
axes[0].set_ylabel('R² (test)')
axes[0].set_title('Width Sweep')
axes[0].grid(alpha=0.3)

axes[1].plot(depths, depth_r2, marker='s', color='coral')
axes[1].set_xticks(depths)
axes[1].set_xlabel('Number of hidden layers (64 units each)')
axes[1].set_ylabel('R² (test)')
axes[1].set_title('Depth Sweep')
axes[1].grid(alpha=0.3)

plt.suptitle('Depth vs Width: R² on California Housing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Width R² values:", [f"{r:.3f}" for r in width_r2])
print("Depth R² values:", [f"{r:.3f}" for r in depth_r2])

## Trap 1 — Wrong Output Activation (Regression)

Using **sigmoid** on the output for regression caps all predictions in (0, 1). House values range up to 5.0 ($500k) — sigmoid will systematically clip everything above ~1.

**Correct choice:** linear output (no activation).

In [ ]:
def forward_with_sigmoid_output(X):
    """
    TODO #8: Implement `forward_with_sigmoid_output()`.

    Steps:
    1. Define helper function `forward_with_sigmoid_output()`
    2. Compute `y_sigmoid` using `forward()`
    3. Plot results -- call `output()`
    4. Plot results -- call `suptitle()`

    Hint:
    axes = plt.subplots(???)
    mx = y_test.min(???)

    Returns: 1 / (1 + np.exp(-z_out))   # sigmoid ...
    """
    raise NotImplementedError("TODO: implement forward_with_sigmoid_output()")

## Trap 2 — Zero Initialisation (Symmetry)

If all weights are zero, every neuron in a layer receives the **same gradient** and updates identically — the entire layer effectively has width 1 regardless of how many units you declare.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `W1_zero` using `zeros()`
# 2. Compute `x_single` using `shape()`
# 3. Compute `h1_zero` using `relu()`
# 4. Call `allclose()` to produce the result
# 5. Plot results -- call `bar()`
#
# Hint:
#    W1_zero = np.zeros(???)
#    b1_zero = np.zeros(???)
#    axes = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Demo: zero-init vs He-init — check that all neurons in layer 1 are identical
W1_zero = np.zeros((8, 128))
b1_zero = np.zeros(128)

# Forward one sample
x_single = X_test_s[:1]   # shape (1, 8)

h1_zero = relu(x_single @ W1_zero + b1_zero)   # (1, 128)
h1_he   = relu(x_single @ W1 + b1)              # (1, 128)

print("Zero-init — are all 128 neurons identical?")
print(f"  All same value: {np.allclose(h1_zero, h1_zero[0, 0])}")
print(f"  Unique activations: {len(np.unique(h1_zero.round(8)))}")

print("\nHe-init — are all 128 neurons identical?")
print(f"  All same value: {np.allclose(h1_he, h1_he[0, 0])}")
print(f"  Unique activations: {len(np.unique(h1_he.round(8)))}")

# Visualise the neuron activation distributions
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].bar(range(20), h1_zero[0, :20], color='red', alpha=0.7)
axes[0].set_title('Zero init — first 20 neuron activations (all identical)')
axes[0].set_xlabel('Neuron index'); axes[0].set_ylabel('Activation')
axes[1].bar(range(20), h1_he[0, :20], color='steelblue', alpha=0.7)
axes[1].set_title('He init — first 20 neuron activations (diverse)')
axes[1].set_xlabel('Neuron index'); axes[1].set_ylabel('Activation')
plt.tight_layout()
plt.show()

## Exercises

**Exercise 1 — Activation ablation**
Replace the hidden activation from `relu` to `tanh` in `MLPRegressor`. Does R² improve or decline on housing?

**Exercise 2 — Feature importance via weight magnitude**
After fitting the MLP, the first-layer weight matrix `mlp.coefs_[0]` has shape (8, 128). Compute the mean absolute weight per input feature and plot a bar chart. Which feature is most predictive?

**Exercise 3 — Unscaled input trap**
Fit an MLPRegressor **without** calling `StandardScaler` first. Compare R² to the scaled version and plot residuals. What changes?

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Fit the model -- call `R²()`
#
# Hint:
#    mlp_tanh = MLPRegressor(hidden_layer_sizes=???)
#    mlp_tanh.fit(???)
#    mlp_tanh.predict(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Exercise 1 scaffold — activation ablation
# Replace 'relu' with 'tanh' and compare R²

mlp_tanh = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',      # <-- change to 'tanh'
    solver='adam',
    max_iter=300,
    random_state=42,
)
mlp_tanh.fit(X_train_s, y_train)
r2_tanh = r2_score(y_test, mlp_tanh.predict(X_test_s))
print(f"R² (relu): {r2_mlp:.4f}")
print(f"R² (tanh): {r2_tanh:.4f}")
# TODO: change activation and observe difference

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `W_first` using `mean()`
#
# Hint:
#    importance = np.abs(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Exercise 2 scaffold — feature importance via first-layer weight magnitude
# mlp.coefs_[0] has shape (n_input_features, n_hidden_units_layer_1)

W_first = mlp.coefs_[0]                         # (8, 128)
importance = np.abs(W_first).mean(axis=1)        # mean absolute weight per input feature

# TODO: plot a horizontal bar chart of importance vs feature_names
# Hint: plt.barh(feature_names, importance[sorted_idx])

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Fit the model -- call `R²()`
#
# Hint:
#    mlp_unscaled = MLPRegressor(hidden_layer_sizes=???)
#    mlp_unscaled.fit(???)
#    mlp_unscaled.predict(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Exercise 3 scaffold — unscaled input trap
# Fit MLPRegressor on raw (un-standardised) X_train and measure R²

mlp_unscaled = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42,
)
mlp_unscaled.fit(X_train, y_train)   # raw features, no StandardScaler
r2_unscaled = r2_score(y_test, mlp_unscaled.predict(X_test))
print(f"R² (scaled):   {r2_mlp:.4f}")
print(f"R² (unscaled): {r2_unscaled:.4f}")
# TODO: plot residuals for both and compare distributions

### What happens if we squeeze the hidden layers too hard?

A very narrow hidden layer forces the network to compress everything through a tiny bottleneck. Try three widths and observe how validation MAE degrades as the bottleneck tightens.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

# Assume X, y are already loaded (California Housing)
scaler = StandardScaler()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

for hidden in [(64, 32), (8, 4), (2, 1)]:
    mlp = MLPRegressor(hidden_layer_sizes=hidden, max_iter=500, random_state=42)
    mlp.fit(X_train_s, y_train)
    mae = mean_absolute_error(y_val, mlp.predict(X_val_s))
    print(f"hidden={hidden}  val MAE={mae:,.0f}")